# Python and Sionna RT Environment Config

In [ ]:
import importlib
import importlib.util
import inspect
import random
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from tqdm import tqdm
import sionna
import sionna.rt
from sionna.rt import (
    load_scene, PlanarArray, Transmitter, Receiver, Camera,
    PathSolver, RadioMapSolver, subcarrier_frequencies
)

import SceneConfigSionna2
import BeamformingCalc
from SceneConfigSionna2 import SceneConfigSionna
from BeamformingCalc import svd_bf


from ntn_music_detection import (
    detect_ntn_music_from_hi,
    noise_subspace_from_music_out,
    build_steering_bank,
    estimate_angle_from_channel_scan,
    angle_error_metrics,
    aod_to_aoa_reverse,
    zenith_to_elevation_deg,
    collapse_cir_to_narrowband,
    sector_index_from_tx_index,
    sector_local_aod_to_global,
    build_true_pair_map,
    run_music_angle_pipeline,
)

# Reload local project modules
importlib.reload(SceneConfigSionna2)
importlib.reload(BeamformingCalc)






In [ ]:
scene = load_scene("blender_scene_big/10km_times_10km/10km_times_10km.xml")
# scene = load_scene("blender_scene_big/26km_times_15km/26km_times_15km_sionna.xml")
# scene = load_scene("Denver_Scene/blouder_plane_itu3/boulder_plane_itu.xml")


Parmeters

In [ ]:
import importlib
import SceneConfigSionna2
importlib.reload(SceneConfigSionna2)

from SceneConfigSionna2 import SceneConfigSionna
SceneConfig = SceneConfigSionna(scene)
SceneConfig.build_coverage_map(grid_size=10, show_xy=True, plot=False)

In [ ]:
ntn_rx=100
tn_rx=100
bs_row = 1
bs_col = 1
nbs = bs_row*bs_col
azimuth = np.random.uniform(0, 360)
elevation = np.random.uniform(35, 90)
fc = 9.99e9  # 
nsect = 3
tx_antenna_rows = 8
tx_antenna_cols = 8
tn_rx_antenna_rows = 1
tn_rx_antenna_cols = 1
tx_antennas = tx_antenna_rows*tx_antenna_cols
tn_antennas = tn_rx_antenna_rows*tn_rx_antenna_cols



In [ ]:
SceneConfig.compute_positions(
    ntn_rx=ntn_rx,
    tn_rx=tn_rx,
    azimuth=azimuth,
    elevation=elevation,
    centerBS=True,
    bs_grid=(bs_row, bs_col),
    bs_boundary=3000,
    tn_building_ratio="sector",   # 0.5 -> 随机且50%在building
    # tn_building_ratio="0.6",   # 0.5 -> 随机且50%在building
    tn_distance=400,
    ntn_building_ratio=0.7,
    plot_grid=True,
    plot_bs=True,
    plot_tn=True,
    plot_ntn=True
)
tn_pos = SceneConfig.tn_pos
tx_pos = SceneConfig.tx_pos
rx_ntn_pos = SceneConfig.rx_ntn_pos


Run Simu

In [ ]:
SceneConfig.compute_paths(
    nsect=nsect,
    fc=fc,
    tx_rows=tx_antenna_rows,
    tx_cols=tx_antenna_cols,
    tn_rx_rows=tn_rx_antenna_rows,
    tn_rx_cols=tn_rx_antenna_cols,
    max_depth=0,
    bandwidth=100e6,
    tx_power_dbm=30
    
)
tn_bs_index = SceneConfig.tn_bs_index
tn_sector_index = SceneConfig.tn_sector_index
cir_tn = SceneConfig.a_tn
cir_ntn = SceneConfig.a_ntn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def _to_np(x):
    return x.numpy() if hasattr(x, "numpy") else np.asarray(x)


def _canonical_rx_tx_paths(arr, num_tx):
    # -> [num_rx, num_tx, num_paths_flat]
    a = np.squeeze(_to_np(arr))
    if a.ndim < 2:
        raise ValueError(f"Need >=2 dims for [rx,tx], got {a.shape}")
    tx_axes = [i for i, s in enumerate(a.shape) if s == num_tx]
    if not tx_axes:
        raise ValueError(f"Cannot find tx-axis size={num_tx} in {a.shape}")
    tx_axis = tx_axes[1] if (tx_axes[0] == 0 and len(tx_axes) > 1) else tx_axes[0]
    a = np.moveaxis(a, [0, tx_axis], [0, 1])
    return a.reshape(a.shape[0], a.shape[1], -1)


def _pair_and_path_power(cir, num_tx, num_paths):
    # Keep singleton dims (num_paths can be 1)
    h = _to_np(cir)
    tx_axes = [i for i, s in enumerate(h.shape) if s == num_tx]
    if not tx_axes:
        raise ValueError(f"Cannot find tx-axis size={num_tx} in cir shape={h.shape}")
    tx_axis = tx_axes[1] if (tx_axes[0] == 0 and len(tx_axes) > 1) else tx_axes[0]

    h_rt = np.moveaxis(h, [0, tx_axis], [0, 1])  # [rx, tx, ...]
    pwr = np.abs(h_rt) ** 2
    pair_power = pwr.sum(axis=tuple(range(2, pwr.ndim)))

    rest = h_rt.shape[2:]
    path_cands = [i for i, s in enumerate(rest) if s == num_paths]
    if not path_cands:
        if num_paths == 1:
            # Fallback: path axis collapsed upstream, treat as single-path
            return pair_power, pair_power[..., None]
        raise ValueError(f"Cannot align path axis: angle_paths={num_paths}, cir_rest={rest}")

    # If multiple singleton candidates, avoid last axis (often time=1) when possible
    pick = path_cands[-1]
    if len(path_cands) > 1 and pick == len(rest) - 1:
        pick = path_cands[-2]

    path_axis = 2 + pick
    sum_axes = tuple(ax for ax in range(2, h_rt.ndim) if ax != path_axis)
    path_power = pwr.sum(axis=sum_axes)
    return pair_power, path_power


# ===== Common config =====
nsect_eff = int(getattr(SceneConfig, "nsect", nsect))
nbs_eff = int(getattr(SceneConfig, "nbs", 1))
num_tx_total = int(nbs_eff * nsect_eff)
if "sionna_phi_is_global" not in globals():
    sionna_phi_is_global = True
if "theta_display_mode" not in globals():
    theta_display_mode = "elevation"  # or "zenith"
theta_mode = str(theta_display_mode).strip().lower()
if theta_mode not in ("zenith", "elevation"):
    raise ValueError(f"theta_display_mode must be 'zenith' or 'elevation', got {theta_display_mode}")
theta_name = "elev" if theta_mode == "elevation" else "theta"
def _theta_show(theta_zenith_deg):
    arr = np.asarray(theta_zenith_deg, dtype=float)
    return 90.0 - arr if theta_mode == "elevation" else arr
angle_eps_deg = 1e-9
channel_eps = 1e-18


# ===== NTN: keep (angle!=0) & (channel!=0), then pick strongest path =====
paths_ntn = SceneConfig.paths_ntn
if not hasattr(paths_ntn, "phi_t"):
    raise AttributeError("paths_ntn.phi_t not found in your Sionna version")

phi_probe = np.squeeze(_to_np(paths_ntn.phi_t))
if num_tx_total not in phi_probe.shape:
    nb_guess = int(nbs * nsect)
    if nb_guess in phi_probe.shape:
        num_tx_total = nb_guess
    elif nsect_eff in phi_probe.shape:
        num_tx_total = nsect_eff
    elif phi_probe.ndim >= 2:
        num_tx_total = int(phi_probe.shape[1])

print(
    f"num_tx_total used={num_tx_total}, SceneConfig.nbs={nbs_eff}, "
    f"SceneConfig.nsect={nsect_eff}, phi_t raw shape={phi_probe.shape}"
)

phi_t = _canonical_rx_tx_paths(paths_ntn.phi_t, num_tx_total)
theta_t = _canonical_rx_tx_paths(paths_ntn.theta_t, num_tx_total)
phi_t_raw_deg = np.mod(np.rad2deg(phi_t), 360.0)
theta_t_deg = np.rad2deg(theta_t)
tx_sector_map = sector_index_from_tx_index(np.arange(num_tx_total), nsect_eff)
if sionna_phi_is_global:
    phi_t_deg = phi_t_raw_deg
else:
    phi_t_deg = sector_local_aod_to_global(
        phi_t_raw_deg,
        sector_index=np.broadcast_to(tx_sector_map[None, :, None], phi_t_raw_deg.shape),
        nsect=nsect_eff,
    )
print("Sionna phi convention:", "global" if sionna_phi_is_global else "sector-local->global")

pair_power_ntn, path_power_ntn = _pair_and_path_power(cir_ntn, num_tx_total, phi_t_deg.shape[2])
angle_nonzero_pair_mask = np.any((np.abs(phi_t_deg) > angle_eps_deg) | (np.abs(theta_t_deg) > angle_eps_deg), axis=2)
channel_nonzero_pair_mask = pair_power_ntn > channel_eps
valid_pair_mask = angle_nonzero_pair_mask & channel_nonzero_pair_mask

ntn_rx_tx_pairs_angle_nonzero = np.argwhere(angle_nonzero_pair_mask)
ntn_rx_tx_pairs_channel_nonzero = np.argwhere(channel_nonzero_pair_mask)
ntn_rx_tx_pairs_valid = np.argwhere(valid_pair_mask)

print("rx-tx pairs with non-zero angle:", ntn_rx_tx_pairs_angle_nonzero.shape[0])
print("rx-tx pairs with non-zero channel:", ntn_rx_tx_pairs_channel_nonzero.shape[0])
print("rx-tx pairs kept by BOTH:", ntn_rx_tx_pairs_valid.shape[0])
print("angle-only pairs:", np.argwhere(angle_nonzero_pair_mask & (~channel_nonzero_pair_mask)).shape[0])
print("channel-only pairs:", np.argwhere((~angle_nonzero_pair_mask) & channel_nonzero_pair_mask).shape[0])

if ntn_rx_tx_pairs_valid.size:
    rx_idx_valid = ntn_rx_tx_pairs_valid[:, 0]
    t_idx_valid = ntn_rx_tx_pairs_valid[:, 1]
    pair_best_path = np.argmax(path_power_ntn[rx_idx_valid, t_idx_valid, :], axis=1)
    pair_phi_t_deg = phi_t_deg[rx_idx_valid, t_idx_valid, pair_best_path]
    pair_theta_t_deg = theta_t_deg[rx_idx_valid, t_idx_valid, pair_best_path]
    pair_bs_idx = t_idx_valid // nsect_eff
    pair_sector_idx = t_idx_valid % nsect_eff
else:
    rx_idx_valid = np.empty((0,), dtype=int)
    t_idx_valid = np.empty((0,), dtype=int)
    pair_best_path = np.empty((0,), dtype=int)
    pair_phi_t_deg = np.empty((0,), dtype=float)
    pair_theta_t_deg = np.empty((0,), dtype=float)
    pair_bs_idx = np.empty((0,), dtype=int)
    pair_sector_idx = np.empty((0,), dtype=int)

ntn_pair_rx_idx_keep = rx_idx_valid
ntn_pair_t_keep = t_idx_valid
ntn_pair_bs_keep = pair_bs_idx
ntn_pair_sector_keep = pair_sector_idx
ntn_pair_path_keep = pair_best_path
ntn_pair_phi_t_deg_keep = pair_phi_t_deg
ntn_pair_theta_t_deg_keep = pair_theta_t_deg

phi_t_ntn_deg = phi_t_deg
theta_t_ntn_deg = theta_t_deg

print(f"\nFirst 20 kept NTN pairs (rx, t, bs, sec, phi_t, {theta_name}):")
for i in range(min(20, len(ntn_pair_rx_idx_keep))):
    theta_show_i = float(_theta_show(ntn_pair_theta_t_deg_keep[i]))
    print(
        f"pair[{i:2d}] rx={int(ntn_pair_rx_idx_keep[i]):3d}, t={int(ntn_pair_t_keep[i]):2d}, "
        f"(bs={int(ntn_pair_bs_keep[i])}, sec={int(ntn_pair_sector_keep[i])}), "
        f"phi_t={float(ntn_pair_phi_t_deg_keep[i]):7.2f} deg, "
        f"{theta_name}={theta_show_i:7.2f} deg"
    )


# ===== TN: for each TN, read serving (bs,sector) angle on strongest path =====
paths_tn = SceneConfig.paths_tn
if not hasattr(paths_tn, "phi_t"):
    raise AttributeError("paths_tn.phi_t not found in your Sionna version")

phi_tn = _canonical_rx_tx_paths(paths_tn.phi_t, num_tx_total)
theta_tn = _canonical_rx_tx_paths(paths_tn.theta_t, num_tx_total)
phi_tn_raw_deg = np.mod(np.rad2deg(phi_tn), 360.0)
theta_t_tn_deg = np.rad2deg(theta_tn)
if sionna_phi_is_global:
    phi_t_tn_deg = phi_tn_raw_deg
else:
    phi_t_tn_deg = sector_local_aod_to_global(
        phi_tn_raw_deg,
        sector_index=np.broadcast_to(tx_sector_map[None, :, None], phi_tn_raw_deg.shape),
        nsect=nsect_eff,
    )

pair_power_tn, path_power_tn = _pair_and_path_power(cir_tn, num_tx_total, phi_t_tn_deg.shape[2])

tn_rx_all = np.arange(tn_pos.shape[0], dtype=int)
tn_t_all = tn_bs_index.astype(int) * nsect_eff + tn_sector_index.astype(int)
tn_in_range = (tn_t_all >= 0) & (tn_t_all < num_tx_total)

tn_rx = tn_rx_all[tn_in_range]
tn_t = tn_t_all[tn_in_range]
tn_bs = tn_bs_index.astype(int)[tn_in_range]
tn_sec = tn_sector_index.astype(int)[tn_in_range]

if tn_rx.size:
    tn_channel_ok = pair_power_tn[tn_rx, tn_t] > channel_eps
    tn_angle_ok = np.any(
        (np.abs(phi_t_tn_deg[tn_rx, tn_t, :]) > angle_eps_deg)
        | (np.abs(theta_t_tn_deg[tn_rx, tn_t, :]) > angle_eps_deg),
        axis=1,
    )
    tn_valid = tn_channel_ok & tn_angle_ok

    tn_best_path_all = np.argmax(path_power_tn[tn_rx, tn_t, :], axis=1)
    tn_phi_all = phi_t_tn_deg[tn_rx, tn_t, tn_best_path_all]
    tn_theta_all = theta_t_tn_deg[tn_rx, tn_t, tn_best_path_all]

    tn_pair_rx_idx_keep = tn_rx[tn_valid]
    tn_pair_t_keep = tn_t[tn_valid]
    tn_pair_bs_keep = tn_bs[tn_valid]
    tn_pair_sector_keep = tn_sec[tn_valid]
    tn_pair_path_keep = tn_best_path_all[tn_valid]
    tn_pair_phi_t_deg_keep = tn_phi_all[tn_valid]
    tn_pair_theta_t_deg_keep = tn_theta_all[tn_valid]
else:
    tn_pair_rx_idx_keep = np.empty((0,), dtype=int)
    tn_pair_t_keep = np.empty((0,), dtype=int)
    tn_pair_bs_keep = np.empty((0,), dtype=int)
    tn_pair_sector_keep = np.empty((0,), dtype=int)
    tn_pair_path_keep = np.empty((0,), dtype=int)
    tn_pair_phi_t_deg_keep = np.empty((0,), dtype=float)
    tn_pair_theta_t_deg_keep = np.empty((0,), dtype=float)

print("\nTN kept links:", len(tn_pair_rx_idx_keep))
print(f"First 20 TN links (rx, t, bs, sec, phi_t, {theta_name}):")
for i in range(min(20, len(tn_pair_rx_idx_keep))):
    theta_show_i = float(_theta_show(tn_pair_theta_t_deg_keep[i]))
    print(
        f"tn[{i:2d}] rx={int(tn_pair_rx_idx_keep[i]):3d}, t={int(tn_pair_t_keep[i]):2d}, "
        f"(bs={int(tn_pair_bs_keep[i])}, sec={int(tn_pair_sector_keep[i])}), "
        f"phi_t={float(tn_pair_phi_t_deg_keep[i]):7.2f} deg, "
        f"{theta_name}={theta_show_i:7.2f} deg"
    )




In [ ]:

# --- Params for post-processing ---
EkT = -174    # Noise energy in dBm/Hz
B = 100e6
Tx_power_dbm = 30
Tx_power = 10 ** ((Tx_power_dbm - 30) / 10)

NF = 7
NF_vsat = 3
NF_bs = 2
N0_dBm = EkT + 10 * np.log10(B) + NF
N0 = 10 ** ((N0_dBm - 30) / 10)
N0_vsat = 10 ** ((EkT + 10 * np.log10(B) + NF_vsat - 30) / 10)
N0_bs = 10 ** ((EkT + 10 * np.log10(B) + NF_bs - 30) / 10)

time = 20e-6
N0_sigma = N0_vsat / Tx_power / time

lambda_ranges = [0, 1e1, 1e3, 1e5]

snr_threshold = -6
inr_threshold = -6
h_ntn_th = np.sqrt(10**(inr_threshold/10) * N0_bs * tx_antennas / Tx_power)
h_tn_th = np.sqrt(10**(snr_threshold/10) * N0_bs * tx_antennas / Tx_power)

# --- MUSIC detection params (NTN) ---
music_num_sources = None
music_threshold = 3.0
music_covariance_mode = "sample"     # "analytic" or "sample"
music_num_snapshots = 100
music_noise_var = N0_bs / Tx_power
music_rng_seed = 7
music_source_estimation = "mdl"      # "mdl" or "energy"
music_energy_ratio = 0.95
music_reduce_ntn_ant = "max"         # "max" or "mean"
music_user_powers = None

# --- Sionna angle convention handling ---
# In your setup, Sionna AOD is treated as global azimuth directly.
# Set False only if you confirm phi_t is sector-local.
sionna_phi_is_global = True
# theta display mode: "zenith" (0=+z) or "elevation" (0=horizontal)
theta_display_mode = "elevation"

# --- MUSIC hat-angle / geometry debug params ---
# channel-mode candidates: "raw" | "conj" | "auto"
music_hat_channel_mode = "conj"
# true-reference for auto selection scoring: "aod" | "aoa_reverse"
music_ref_mode_for_auto = "aod"
# minimum matched pairs required for candidate scoring
music_auto_mode_min_pairs = 1

# include BS sector yaw/pitch/roll in steering
music_use_sector_orientation = True
music_sector_pitch_rad = -0.174533
music_sector_roll_rad = 0.0
music_rotation_order = "zyx"

# manifold candidates:
# - fixed: e.g. "yz:+1"
# - auto: use list in music_manifold_auto_list
music_manifold_mode = "auto"
music_manifold_auto_list = ["yz:+1", "yz:-1", "xz:+1", "xz:-1"]

# steering vector flatten order candidates for array indexing
# "auto" -> try list below; otherwise use "C" or "F"
music_flatten_mode = "F"  # Sionna PlanarArray is column-first
music_flatten_mode_list = ["C", "F"]

# channel-to-steering matching mode
# "complex" uses full complex vector; "phase_only" ignores amplitude
# "auto" -> try list below
music_scan_mode = "auto"
music_scan_mode_list = ["complex", "phase_only"]

# azimuth output offset candidates (deg) for convention mismatch correction
# "auto" -> try list below; otherwise use numeric scalar (e.g., 90.0)
music_phi_offset_mode = "auto"
music_phi_offset_list = [0.0, 90.0, 180.0, 270.0]

# sector forward-halfspace constraint in peak search
music_sector_forward_only = True
music_sector_forward_cos_min = 0.0

# ===== Post-processing (beamforming / SNR / INR) =====

h_tn_all = collapse_cir_to_narrowband(cir_tn)
h_ntn_all = collapse_cir_to_narrowband(cir_ntn)

interfered_ntn = np.count_nonzero(np.any(h_ntn_all != 0, axis=(2, 3)))





In [ ]:

eps = 1e-12  # avoid log10(0)
snr_list = []
h_ntn_dB_list = []
h_tn_dB_list = []

w_t_record = []
v_null_record = {l: [] for l in lambda_ranges}
v_null_t_index = {l: [] for l in lambda_ranges}

eigen_value_dict = {l: [] for l in lambda_ranges}
snr_nulling_dict = {l: [] for l in lambda_ranges}
snr_nulling_hat_dict = {l: [] for l in lambda_ranges}
snr_degration_dict = {l: [] for l in lambda_ranges}
inr_nulling_dict = {l: [] for l in lambda_ranges}
inr_nulling_hat_dict = {l: [] for l in lambda_ranges}

best_tx_list = []
w_t_grouped, w_r_grouped, h_grouped = {}, {}, {}

# MUSIC debug records
music_score_user_record = []
music_score_per_ant_record = []
music_num_sources_record = []

# key=(rx_idx, t_idx)
music_detected_pair_hat = {}

# diagnostics for auto selection
music_candidate_metric_log = {}
music_selected_metric_log = []
music_hat_mode_selected_record = []
music_manifold_selected_record = []
music_flatten_selected_record = []
music_scan_selected_record = []
music_phi_offset_selected_record = []

# Optional MUSIC AoA grid defaults
if "music_phi_grid_deg" not in globals():
    music_phi_grid_deg = np.arange(0.0, 360.0, 2.0)
if "music_theta_grid_deg" not in globals():
    music_theta_grid_deg = np.arange(0.0, 181.0, 2.0)
if "music_peak_top_n" not in globals():
    music_peak_top_n = 24
if "music_peak_min_sep_phi" not in globals():
    music_peak_min_sep_phi = 4.0
if "music_peak_min_sep_theta" not in globals():
    music_peak_min_sep_theta = 4.0

if "music_hat_channel_mode" not in globals():
    music_hat_channel_mode = "conj"
if "music_ref_mode_for_auto" not in globals():
    music_ref_mode_for_auto = "aod"
if "sionna_phi_is_global" not in globals():
    sionna_phi_is_global = True
if "theta_display_mode" not in globals():
    theta_display_mode = "elevation"
if "music_auto_mode_min_pairs" not in globals():
    music_auto_mode_min_pairs = 1
if "music_use_sector_orientation" not in globals():
    music_use_sector_orientation = True
if "music_sector_pitch_rad" not in globals():
    music_sector_pitch_rad = -0.174533
if "music_sector_roll_rad" not in globals():
    music_sector_roll_rad = 0.0
if "music_rotation_order" not in globals():
    music_rotation_order = "zyx"
if "music_manifold_mode" not in globals():
    music_manifold_mode = "auto"
if "music_manifold_auto_list" not in globals():
    music_manifold_auto_list = ["yz:+1", "yz:-1", "xz:+1", "xz:-1"]
if "music_sector_forward_only" not in globals():
    music_sector_forward_only = True
if "music_sector_forward_cos_min" not in globals():
    music_sector_forward_cos_min = 0.0
if "music_flatten_mode" not in globals():
    music_flatten_mode = "F"
if "music_flatten_mode_list" not in globals():
    music_flatten_mode_list = ["C", "F"]
if "music_scan_mode" not in globals():
    music_scan_mode = "auto"
if "music_scan_mode_list" not in globals():
    music_scan_mode_list = ["complex", "phase_only"]
if "music_phi_offset_mode" not in globals():
    music_phi_offset_mode = "auto"
if "music_phi_offset_list" not in globals():
    music_phi_offset_list = [0.0, 90.0, 180.0, 270.0]

def _parse_manifold_label(lbl):
    txt = str(lbl).strip().lower().replace(" ", "")
    if ":" in txt:
        plane, sgn = txt.split(":", 1)
    else:
        plane, sgn = txt, "+1"

    if plane not in ("yz", "xz", "xy"):
        raise ValueError(f"Invalid manifold plane: {plane}. Use yz/xz/xy")

    if sgn in ("+1", "1", "+"):
        sign = 1
    elif sgn in ("-1", "-"):
        sign = -1
    else:
        raise ValueError(f"Invalid manifold sign: {sgn}. Use +1/-1")

    return f"{plane}:{'+' if sign > 0 else '-'}1", plane, sign


def _build_manifold_candidates():
    if str(music_manifold_mode).lower() == "auto":
        src = list(music_manifold_auto_list)
    else:
        src = [music_manifold_mode]

    out = []
    seen = set()
    for x in src:
        key, plane, sign = _parse_manifold_label(x)
        if key in seen:
            continue
        seen.add(key)
        out.append((key, plane, sign))

    if len(out) == 0:
        out = [("yz:+1", "yz", 1)]
    return out


def _build_flatten_candidates():
    if str(music_flatten_mode).lower() == "auto":
        src = list(music_flatten_mode_list)
    else:
        src = [music_flatten_mode]

    out = []
    seen = set()
    for x in src:
        key = str(x).strip().upper()
        if key not in ("C", "F"):
            continue
        if key in seen:
            continue
        seen.add(key)
        out.append(key)

    if len(out) == 0:
        out = ["C"]
    return out


def _build_scan_candidates():
    if str(music_scan_mode).lower() == "auto":
        src = list(music_scan_mode_list)
    else:
        src = [music_scan_mode]

    out = []
    seen = set()
    for x in src:
        key = str(x).strip().lower()
        if key not in ("complex", "phase_only"):
            continue
        if key in seen:
            continue
        seen.add(key)
        out.append(key)

    if len(out) == 0:
        out = ["complex"]
    return out


def _build_phi_offset_candidates():
    if str(music_phi_offset_mode).lower() == "auto":
        src = list(music_phi_offset_list)
    else:
        src = [music_phi_offset_mode]

    out = []
    seen = set()
    for x in src:
        try:
            val = float(x)
        except Exception:
            continue
        key = float(val % 360.0)
        # round key to 1 decimal for stable ckey formatting
        key = float(np.round(key, 1))
        if key in seen:
            continue
        seen.add(key)
        out.append(key)

    if len(out) == 0:
        out = [0.0]
    return out


# Build true-angle map (from Sionna) if available
sionna_map = {}
if (
    "ntn_pair_rx_idx_keep" in globals()
    and "ntn_pair_t_keep" in globals()
    and "ntn_pair_phi_t_deg_keep" in globals()
    and "ntn_pair_theta_t_deg_keep" in globals()
    and "ntn_pair_bs_keep" in globals()
    and "ntn_pair_sector_keep" in globals()
):
    sionna_map = {
        (int(rx), int(t)): (float(phi), float(theta), int(bs), int(sec))
        for rx, t, phi, theta, bs, sec in zip(
            ntn_pair_rx_idx_keep,
            ntn_pair_t_keep,
            ntn_pair_phi_t_deg_keep,
            ntn_pair_theta_t_deg_keep,
            ntn_pair_bs_keep,
            ntn_pair_sector_keep,
        )
    }

# safer tx count for loop (avoid notebook nbs mismatch)
num_tx_total_eff = int(h_ntn_all.shape[2])
nsect_eff = int(getattr(SceneConfig, "nsect", nsect)) if "SceneConfig" in globals() else int(nsect)

# Use one-to-one TN pairing (nearest BS + sector)
for r in range(tn_pos.shape[0]):
    t = int(tn_bs_index[r]) * nsect + int(tn_sector_index[r])
    h_tn = h_tn_all[r, :, t, :].T
    h_tn_norm = np.linalg.norm(h_tn, ord="fro")
    h_tn_dB = 10 * np.log10(np.maximum(h_tn_norm**2, eps))
    h_tn_dB_list.append(h_tn_dB.item())

    if np.any(h_tn_norm > h_tn_th):
        w_t, w_r = svd_bf(h_tn, tx_antennas)
        snr = 10 * np.log10((np.abs(w_t.conj().T @ h_tn @ w_r) ** 2) * Tx_power / N0)

        best_tx_list.append(t)
        snr_list.append(snr.item())

        if t not in w_t_grouped:
            w_t_grouped[t], w_r_grouped[t], h_grouped[t] = [], [], []
        w_t_grouped[t].append(w_t)
        w_r_grouped[t].append(w_r)
        h_grouped[t].append(h_tn)

for t in w_t_grouped:
    w_t_grouped[t] = np.array(w_t_grouped[t])
    w_r_grouped[t] = np.array(w_r_grouped[t])
    h_grouped[t] = np.array(h_grouped[t])

if len(w_t_grouped) == 0:
    raise ValueError("No valid TN links after thresholding.")

min_count = min(len(w_t_grouped[t]) for t in w_t_grouped)

inr_list = []
ntn_kept_indices = []
mask_record = []

for m in range(min_count):
    valid_mask = np.zeros((ntn_rx,), dtype=bool)
    h_i_gain_sum = np.zeros((ntn_rx,), dtype=np.float64)
    h_i_null_gain_sum_dict = {l: np.zeros((ntn_rx,), dtype=np.float64) for l in lambda_ranges}
    h_i_null_gain_sum_hat_dict = {l: np.zeros((ntn_rx,), dtype=np.float64) for l in lambda_ranges}

    for t in range(num_tx_total_eff):
        if t not in w_t_grouped or len(w_t_grouped[t]) <= m:
            continue

        w_t = w_t_grouped[t][m]
        v_t = np.conj(w_t)
        w_t_record.append(v_t.copy())

        h_i_raw = h_ntn_all[:, :, t, :]
        h_i_norms = np.linalg.norm(h_i_raw, axis=2)
        h_ntn_dB = 10 * np.log10(np.maximum(h_i_norms**2, eps))
        h_ntn_dB_list.append(h_ntn_dB)

        if music_hat_channel_mode in ("raw", "conj"):
            candidate_modes = [music_hat_channel_mode]
        else:
            candidate_modes = ["raw", "conj"]

        manifold_candidates = _build_manifold_candidates()
        candidate_results = {}

        for mode in candidate_modes:
            h_i_music = h_i_raw if mode == "raw" else np.conj(h_i_raw)

            music_out_mode = detect_ntn_music_from_hi(
                hi=h_i_music,
                num_sources=music_num_sources,
                threshold=music_threshold,
                user_powers=music_user_powers,
                noise_var=music_noise_var,
                covariance_mode=music_covariance_mode,
                num_snapshots=music_num_snapshots,
                rng_seed=music_rng_seed,
                source_estimation=music_source_estimation,
                energy_ratio=music_energy_ratio,
                reduce_ntn_ant=music_reduce_ntn_ant,
            )

            detected_mask_user_mode = music_out_mode["detected_mask_user"]
            detected_mask_per_ant_mode = music_out_mode["detected_mask_per_ant"]
            score_user_mode = music_out_mode["score_user"]
            indices_kept_mode = np.where(detected_mask_user_mode)[0]

            en = noise_subspace_from_music_out(music_out_mode)

            m_dim = h_i_music.shape[2]
            nr = int(tx_antenna_rows)
            nc = int(tx_antenna_cols)
            if nr * nc != m_dim:
                nr, nc = 1, m_dim

            sec_idx = int(t) % int(max(nsect_eff, 1))
            if music_use_sector_orientation:
                yaw = 2.0 * np.pi * sec_idx / float(max(nsect_eff, 1))
                orientation_rad = (float(yaw), float(music_sector_pitch_rad), float(music_sector_roll_rad))
            else:
                orientation_rad = (0.0, 0.0, 0.0)

            for mani_label, panel_plane, phase_sign in manifold_candidates:
                flatten_candidates = _build_flatten_candidates()
                scan_candidates = _build_scan_candidates()
                steering_cache = {}

                for flat_mode in flatten_candidates:
                    cache_key = (mani_label, flat_mode)
                    if cache_key not in steering_cache:
                        a_bank, phi_bank_deg, theta_bank_deg = build_steering_bank(
                            num_rows=nr,
                            num_cols=nc,
                            phi_grid_deg=music_phi_grid_deg,
                            theta_grid_deg=music_theta_grid_deg,
                            orientation_rad=orientation_rad,
                            panel_plane=panel_plane,
                            phase_sign=phase_sign,
                            flatten_order=flat_mode,
                            forward_only=bool(music_sector_forward_only),
                            forward_cos_min=float(music_sector_forward_cos_min),
                            rotation_order=music_rotation_order,
                        )
                        steering_cache[cache_key] = (a_bank, phi_bank_deg, theta_bank_deg)

                    a_bank, phi_bank_deg, theta_bank_deg = steering_cache[cache_key]

                    for scan_mode in scan_candidates:
                        phi_offset_candidates = _build_phi_offset_candidates()
                        for phi_offset_deg in phi_offset_candidates:
                            hats_mode = {}
                            for rx_idx in indices_kept_mode:
                                ant_norms = np.linalg.norm(h_i_music[int(rx_idx), :, :], axis=1)
                                ant_idx = int(np.argmax(ant_norms))
                                phi_hat_deg, theta_hat_deg, fit_score = estimate_angle_from_channel_scan(
                                    h_i_music[int(rx_idx), ant_idx, :],
                                    a_bank,
                                    phi_bank_deg,
                                    theta_bank_deg,
                                    scan_mode=scan_mode,
                                )
                                if np.isfinite(phi_hat_deg):
                                    phi_hat_deg = float((phi_hat_deg + float(phi_offset_deg)) % 360.0)
                                hats_mode[(int(rx_idx), int(t))] = (
                                    float(phi_hat_deg),
                                    float(theta_hat_deg),
                                    float(score_user_mode[int(rx_idx)]),
                                )

                            metric_mode = None
                            matched_keys_mode = [
                                (int(rx_idx), int(t))
                                for rx_idx in indices_kept_mode
                                if (int(rx_idx), int(t)) in sionna_map
                            ]

                            if len(matched_keys_mode) >= int(music_auto_mode_min_pairs):
                                phi_true_mode = np.array([sionna_map[k][0] for k in matched_keys_mode], dtype=float)
                                theta_true_mode = np.array([sionna_map[k][1] for k in matched_keys_mode], dtype=float)
                                phi_hat_mode = np.array([hats_mode[k][0] for k in matched_keys_mode], dtype=float)
                                theta_hat_mode = np.array([hats_mode[k][1] for k in matched_keys_mode], dtype=float)

                                metric_mode = angle_error_metrics(
                                    phi_true_mode,
                                    theta_true_mode,
                                    phi_hat_mode,
                                    theta_hat_mode,
                                    reference_mode=music_ref_mode_for_auto,
                                )
                                metric_mode["mode"] = mode
                                metric_mode["manifold"] = mani_label
                                metric_mode["flatten"] = flat_mode
                                metric_mode["scan"] = scan_mode
                                metric_mode["phi_offset_deg"] = float(phi_offset_deg)
                                metric_mode["t"] = int(t)
                                metric_mode["m"] = int(m)

                                ckey = f"{mode}|{mani_label}|{flat_mode}|{scan_mode}|{float(phi_offset_deg):.1f}"
                                if ckey not in music_candidate_metric_log:
                                    music_candidate_metric_log[ckey] = []
                                music_candidate_metric_log[ckey].append(metric_mode)

                            ckey = f"{mode}|{mani_label}|{flat_mode}|{scan_mode}|{float(phi_offset_deg):.1f}"
                            candidate_results[ckey] = {
                                "mode": mode,
                                "manifold": mani_label,
                                "flatten": flat_mode,
                                "scan": scan_mode,
                                "phi_offset_deg": float(phi_offset_deg),
                                "music_out": music_out_mode,
                                "detected_mask_user": detected_mask_user_mode,
                                "detected_mask_per_ant": detected_mask_per_ant_mode,
                                "score_user": score_user_mode,
                                "hats": hats_mode,
                                "metric": metric_mode,
                            }

        # choose selected candidate
        if music_hat_channel_mode in ("raw", "conj") and str(music_manifold_mode).lower() != "auto" and str(music_flatten_mode).lower() != "auto" and str(music_scan_mode).lower() != "auto" and str(music_phi_offset_mode).lower() != "auto":
            fixed_key, _, _ = _parse_manifold_label(music_manifold_mode)
            fixed_flat = str(music_flatten_mode).upper()
            fixed_scan = str(music_scan_mode).lower()
            fixed_poff = float(np.round(float(music_phi_offset_mode) % 360.0, 1))
            selected_key = f"{music_hat_channel_mode}|{fixed_key}|{fixed_flat}|{fixed_scan}|{fixed_poff:.1f}"
            if selected_key not in candidate_results:
                selected_key = list(candidate_results.keys())[0]
        else:
            scored = []
            for ckey, cres in candidate_results.items():
                met = cres["metric"]
                if met is None:
                    continue
                val = float(met["phi_mae_deg"] + met["theta_mae_deg"])
                scored.append((val, ckey))

            if len(scored) > 0:
                scored.sort(key=lambda x: x[0])
                selected_key = scored[0][1]
            else:
                # fallback: prefer raw + first manifold
                pref = []
                for ckey, cres in candidate_results.items():
                    rank_mode = 0 if cres["mode"] == "raw" else 1
                    rank_mani = 0
                    rank_flat = 0 if cres.get("flatten", "C") == "C" else 1
                    rank_scan = 0 if cres.get("scan", "complex") == "complex" else 1
                    rank_poff = 0 if abs(float(cres.get("phi_offset_deg", 0.0))) < 1e-9 else 1
                    pref.append((rank_mode, rank_mani, rank_flat, rank_scan, rank_poff, ckey))
                pref.sort()
                selected_key = pref[0][5]

        selected = candidate_results[selected_key]
        music_hat_mode_selected_record.append(selected["mode"])
        music_manifold_selected_record.append(selected["manifold"])
        music_flatten_selected_record.append(selected["flatten"])
        music_scan_selected_record.append(selected["scan"])
        music_phi_offset_selected_record.append(float(selected.get("phi_offset_deg", 0.0)))

        if selected["metric"] is not None:
            music_selected_metric_log.append(selected["metric"])

        detected_mask_user = selected["detected_mask_user"]
        detected_mask_per_ant = selected["detected_mask_per_ant"]
        score_user = selected["score_user"]
        hats_selected = selected["hats"]

        music_score_user_record.append(score_user)
        music_score_per_ant_record.append(selected["music_out"]["score_per_ant"])
        music_num_sources_record.append(int(selected["music_out"]["num_sources_est"]))

        mask = ~detected_mask_per_ant
        valid_mask |= detected_mask_user
        mask_record.append(mask.copy())

        indices_kept = np.where(detected_mask_user)[0]
        ntn_kept_indices.append(indices_kept)

        for rx_idx in indices_kept:
            key = (int(rx_idx), int(t))
            if key in hats_selected:
                phi_hat_deg, theta_hat_deg, sc = hats_selected[key]
            else:
                phi_hat_deg, theta_hat_deg, sc = np.nan, np.nan, float(score_user[int(rx_idx)])

            prev = music_detected_pair_hat.get(key)
            if prev is None or sc > prev["score"]:
                music_detected_pair_hat[key] = {
                    "score": sc,
                    "bs": int(t) // int(max(nsect_eff, 1)),
                    "sec": int(t) % int(max(nsect_eff, 1)),
                    "phi_hat_deg": float(phi_hat_deg),
                    "theta_hat_deg": float(theta_hat_deg),
                    "hat_mode": selected["mode"],
                    "manifold": selected["manifold"],
                    "flatten": selected["flatten"],
                    "scan": selected["scan"],
                    "phi_offset_deg": float(selected.get("phi_offset_deg", 0.0)),
                }

        noise = N0_sigma / np.sqrt(2) * (
            np.random.randn(*h_i_raw.shape) + 1j * np.random.randn(*h_i_raw.shape)
        )
        h_i_hat = h_i_raw + noise

        h_i_det = h_i_raw.copy()
        h_i_hat_det = h_i_hat.copy()
        h_i_det[mask] = 0
        h_i_hat_det[mask] = 0

        interference_term = np.einsum("uri,urj->ij", np.conj(h_i_det), h_i_det)
        interference_term_hat = np.einsum("uri,urj->ij", np.conj(h_i_hat_det), h_i_hat_det)

        h_i_gain = np.abs(np.sum(h_i_det @ w_t, axis=1).squeeze() / np.sqrt(h_i_det.shape[1])) ** 2
        h_i_gain_sum += h_i_gain

    inr = 10 * np.log10(np.maximum(h_i_gain_sum * Tx_power / N0, eps))
    inr_list.extend(inr[valid_mask])

print("ntn_number_has_path_to BS:", interfered_ntn)
print("Used_tn:", min_count)
if len(music_num_sources_record) > 0:
    print("MUSIC estimated K (mean):", float(np.mean(music_num_sources_record)))

if len(music_hat_mode_selected_record) > 0:
    raw_cnt = sum(1 for x in music_hat_mode_selected_record if x == "raw")
    conj_cnt = sum(1 for x in music_hat_mode_selected_record if x == "conj")
    print(f"MUSIC hat mode selected: raw={raw_cnt}, conj={conj_cnt}, total={len(music_hat_mode_selected_record)}")

if len(music_manifold_selected_record) > 0:
    vals, cnts = np.unique(np.asarray(music_manifold_selected_record, dtype=object), return_counts=True)
    msg = ", ".join([f"{v}:{int(c)}" for v, c in zip(vals, cnts)])
    print("MUSIC manifold selected:", msg)
if len(music_flatten_selected_record) > 0:
    vals, cnts = np.unique(np.asarray(music_flatten_selected_record, dtype=object), return_counts=True)
    msg = ", ".join([f"{v}:{int(c)}" for v, c in zip(vals, cnts)])
    print("MUSIC flatten selected:", msg)
if len(music_scan_selected_record) > 0:
    vals, cnts = np.unique(np.asarray(music_scan_selected_record, dtype=object), return_counts=True)
    msg = ", ".join([f"{v}:{int(c)}" for v, c in zip(vals, cnts)])
    print("MUSIC scan selected:", msg)
if len(music_phi_offset_selected_record) > 0:
    vals, cnts = np.unique(np.asarray(music_phi_offset_selected_record, dtype=float), return_counts=True)
    msg = ", ".join([f"{float(v):.1f}:{int(c)}" for v, c in zip(vals, cnts)])
    print("MUSIC phi-offset selected:", msg)

unique_ntn_kept_indices = (
    np.unique(np.concatenate(ntn_kept_indices)) if ntn_kept_indices else np.array([], dtype=int)
)
left_ntn_pos = (
    rx_ntn_pos[unique_ntn_kept_indices] if unique_ntn_kept_indices.size else np.empty((0, 3))
)
print("detected ntn number:", unique_ntn_kept_indices.shape)

if len(music_detected_pair_hat) > 0:
    det_pairs = sorted(music_detected_pair_hat.keys(), key=lambda k: (k[1], k[0]))
    music_detected_pair_rx_idx = np.array([k[0] for k in det_pairs], dtype=int)
    music_detected_pair_t_idx = np.array([k[1] for k in det_pairs], dtype=int)
    music_detected_pair_bs_idx = np.array([music_detected_pair_hat[k]["bs"] for k in det_pairs], dtype=int)
    music_detected_pair_sector_idx = np.array([music_detected_pair_hat[k]["sec"] for k in det_pairs], dtype=int)
    music_detected_pair_phi_hat_deg = np.array([music_detected_pair_hat[k]["phi_hat_deg"] for k in det_pairs], dtype=float)
    music_detected_pair_theta_hat_deg = np.array([music_detected_pair_hat[k]["theta_hat_deg"] for k in det_pairs], dtype=float)
    music_detected_pair_hat_mode = np.array([music_detected_pair_hat[k]["hat_mode"] for k in det_pairs], dtype=object)
    music_detected_pair_manifold = np.array([music_detected_pair_hat[k]["manifold"] for k in det_pairs], dtype=object)
    music_detected_pair_flatten = np.array([music_detected_pair_hat[k]["flatten"] for k in det_pairs], dtype=object)
    music_detected_pair_scan = np.array([music_detected_pair_hat[k]["scan"] for k in det_pairs], dtype=object)
    music_detected_pair_phi_offset = np.array([music_detected_pair_hat[k]["phi_offset_deg"] for k in det_pairs], dtype=float)
else:
    music_detected_pair_rx_idx = np.empty((0,), dtype=int)
    music_detected_pair_t_idx = np.empty((0,), dtype=int)
    music_detected_pair_bs_idx = np.empty((0,), dtype=int)
    music_detected_pair_sector_idx = np.empty((0,), dtype=int)
    music_detected_pair_phi_hat_deg = np.empty((0,), dtype=float)
    music_detected_pair_theta_hat_deg = np.empty((0,), dtype=float)
    music_detected_pair_hat_mode = np.empty((0,), dtype=object)
    music_detected_pair_manifold = np.empty((0,), dtype=object)
    music_detected_pair_flatten = np.empty((0,), dtype=object)
    music_detected_pair_scan = np.empty((0,), dtype=object)
    music_detected_pair_phi_offset = np.empty((0,), dtype=float)

print("MUSIC detected pair count (all kept):", len(music_detected_pair_rx_idx))

if len(sionna_map) > 0:
    music_map = {
        (int(rx), int(t)): (float(phi_hat), float(theta_hat), int(bs), int(sec), mode, mani, flat, scan, poff)
        for rx, t, phi_hat, theta_hat, bs, sec, mode, mani, flat, scan, poff in zip(
            music_detected_pair_rx_idx,
            music_detected_pair_t_idx,
            music_detected_pair_phi_hat_deg,
            music_detected_pair_theta_hat_deg,
            music_detected_pair_bs_idx,
            music_detected_pair_sector_idx,
            music_detected_pair_hat_mode,
            music_detected_pair_manifold,
            music_detected_pair_flatten,
            music_detected_pair_scan,
            music_detected_pair_phi_offset,
        )
    }

    hat_pairs = sorted(music_map.keys(), key=lambda k: (k[1], k[0]))

    print("\nDetected pairs (HAT):", len(hat_pairs))
    theta_name = "elev" if str(theta_display_mode).lower() == "elevation" else "theta"
    print(f"format: pair(rx,t,bs,sec,mode,mani,flat,scan,poff) | trueAOD_global(phi,{theta_name}) | hat_global(phi,{theta_name})")

    for i, (rx, t) in enumerate(hat_pairs):
        phi_hat, theta_hat, bs_hat, sec_hat, mode_hat, mani_hat, flat_hat, scan_hat, poff_hat = music_map[(rx, t)]
        if str(theta_display_mode).lower() == "elevation":
            theta_hat_show = float(zenith_to_elevation_deg(np.array([theta_hat]))[0])
        else:
            theta_hat_show = float(theta_hat)

        if (rx, t) in sionna_map:
            phi_true, theta_true, _bs_t, _sec_t = sionna_map[(rx, t)]
            phi_ref = float(phi_true)
            theta_ref = float(theta_true)
            if str(theta_display_mode).lower() == "elevation":
                theta_ref_show = float(zenith_to_elevation_deg(np.array([theta_ref]))[0])
            else:
                theta_ref_show = theta_ref
        else:
            phi_true, theta_true = np.nan, np.nan
            phi_ref, theta_ref = np.nan, np.nan
            theta_ref_show = np.nan

        print(
            f"pair[{i:3d}] (rx={int(rx):3d}, t={int(t):2d}, bs={int(bs_hat)}, sec={int(sec_hat)}, mode={mode_hat}, mani={mani_hat}, flat={flat_hat}, scan={scan_hat}, poff={poff_hat:.1f}) | "
            f"trueAOD_global(phi={phi_ref:7.2f}, {theta_name}={theta_ref_show:7.2f}) | "
            f"hat_global(phi={phi_hat:7.2f}, {theta_name}={theta_hat_show:7.2f})"
        )

    matched_keys = [k for k in hat_pairs if k in sionna_map]
    if len(matched_keys) > 0:
        music_pair_rx_idx_keep = np.array([k[0] for k in matched_keys], dtype=int)
        music_pair_t_keep = np.array([k[1] for k in matched_keys], dtype=int)
        music_pair_bs_keep = np.array([music_map[k][2] for k in matched_keys], dtype=int)
        music_pair_sector_keep = np.array([music_map[k][3] for k in matched_keys], dtype=int)
        music_pair_phi_t_deg_keep = np.array([music_map[k][0] for k in matched_keys], dtype=float)
        music_pair_theta_t_deg_keep = np.array([music_map[k][1] for k in matched_keys], dtype=float)

        phi_true_match = np.array([sionna_map[k][0] for k in matched_keys], dtype=float)
        theta_true_match = np.array([sionna_map[k][1] for k in matched_keys], dtype=float)
        phi_hat_match = np.array([music_map[k][0] for k in matched_keys], dtype=float)
        theta_hat_match = np.array([music_map[k][1] for k in matched_keys], dtype=float)
        if str(theta_display_mode).lower() == "elevation":
            theta_true_metric = zenith_to_elevation_deg(theta_true_match)
            theta_hat_metric = zenith_to_elevation_deg(theta_hat_match)
            theta_metric_name = "elev"
        else:
            theta_true_metric = theta_true_match
            theta_hat_metric = theta_hat_match
            theta_metric_name = "theta"

        metrics_aod = angle_error_metrics(
            phi_true_match,
            theta_true_metric,
            phi_hat_match,
            theta_hat_metric,
            reference_mode="aod",
        )

        print("\nAngle Error Summary on matched detected pairs:")
        print(
            "  ref=AOD(global) : "
            f"N={int(metrics_aod['count'])}, "
            f"phi_MAE={metrics_aod['phi_mae_deg']:.2f} deg, "
            f"{theta_metric_name}_MAE={metrics_aod['theta_mae_deg']:.2f} deg"
        )

        # show best candidate by weighted error over all tested candidates
        def _weighted_metric(logs, key):
            if len(logs) == 0:
                return np.nan
            w = np.array([max(float(m.get("count", 0.0)), 0.0) for m in logs], dtype=float)
            x = np.array([float(m.get(key, np.nan)) for m in logs], dtype=float)
            mask = (~np.isnan(x)) & (w > 0)
            if np.count_nonzero(mask) == 0:
                return np.nan
            return float(np.sum(w[mask] * x[mask]) / np.sum(w[mask]))

        if len(music_candidate_metric_log) > 0:
            print("\nAuto-Selection Diagnostic (weighted over matched pairs):")
            rows = []
            for ckey, logs in music_candidate_metric_log.items():
                total_n = int(np.sum([m.get("count", 0.0) for m in logs]))
                phi_mae = _weighted_metric(logs, "phi_mae_deg")
                theta_mae = _weighted_metric(logs, "theta_mae_deg")
                rows.append((phi_mae + theta_mae, ckey, total_n, phi_mae, theta_mae))
            rows.sort(key=lambda x: x[0])
            print(f"  total candidates with matched pairs: {len(rows)}")
            for score, ckey, total_n, phi_mae, theta_mae in rows:
                print(
                    f"  {ckey:18s}: total_N={total_n:4d}, "
                    f"phi_MAE={phi_mae:7.2f} deg, theta_MAE={theta_mae:7.2f} deg"
                )
    else:
        music_pair_rx_idx_keep = np.empty((0,), dtype=int)
        music_pair_t_keep = np.empty((0,), dtype=int)
        music_pair_bs_keep = np.empty((0,), dtype=int)
        music_pair_sector_keep = np.empty((0,), dtype=int)
        music_pair_phi_t_deg_keep = np.empty((0,), dtype=float)
        music_pair_theta_t_deg_keep = np.empty((0,), dtype=float)
else:
    print("Sionna pair-angle arrays not found yet. Run the pair extraction cell first.")



In [ ]:
# ===== TN MUSIC angle display (generic pipeline call) =====
import numpy as np

# Build true TN map from strongest-path pairs extracted earlier
tn_true_map = build_true_pair_map(
    tn_pair_rx_idx_keep,
    tn_pair_t_keep,
    tn_pair_phi_t_deg_keep,
    tn_pair_theta_t_deg_keep,
    tn_pair_bs_keep,
    tn_pair_sector_keep,
)
tn_pair_keys = sorted(tn_true_map.keys(), key=lambda k: (k[1], k[0]))

nsect_eff_tn = int(globals().get("nsect_eff", getattr(SceneConfig, "nsect", nsect)))

# Shared MUSIC pipeline for TN/NTN (same h_all tensor interface)
tn_music_out = run_music_angle_pipeline(
    h_tn_all,
    tx_rows=int(tx_antenna_rows),
    tx_cols=int(tx_antenna_cols),
    nsect=nsect_eff_tn,
    true_pair_map=tn_true_map,
    pair_keys=tn_pair_keys,  # constrain to serving TN links
    detect_num_sources=music_num_sources,
    detect_threshold=music_threshold,
    detect_user_powers=music_user_powers,
    detect_noise_var=music_noise_var,
    detect_covariance_mode=music_covariance_mode,
    detect_num_snapshots=music_num_snapshots,
    detect_rng_seed=music_rng_seed,
    detect_source_estimation=music_source_estimation,
    detect_energy_ratio=music_energy_ratio,
    detect_reduce_rx_ant=music_reduce_ntn_ant,
    hat_channel_mode=music_hat_channel_mode,
    ref_mode_for_auto=music_ref_mode_for_auto,
    auto_mode_min_pairs=music_auto_mode_min_pairs,
    use_sector_orientation=music_use_sector_orientation,
    sector_pitch_rad=music_sector_pitch_rad,
    sector_roll_rad=music_sector_roll_rad,
    rotation_order=music_rotation_order,
    manifold_mode=music_manifold_mode,
    manifold_auto_list=music_manifold_auto_list,
    flatten_mode=music_flatten_mode,
    flatten_mode_list=music_flatten_mode_list,
    scan_mode=music_scan_mode,
    scan_mode_list=music_scan_mode_list,
    phi_offset_mode=music_phi_offset_mode,
    phi_offset_list=music_phi_offset_list,
    sector_forward_only=music_sector_forward_only,
    sector_forward_cos_min=music_sector_forward_cos_min,
    phi_grid_deg=music_phi_grid_deg,
    theta_grid_deg=music_theta_grid_deg,
)

tn_music_pair_hat = tn_music_out["pair_hat"]
tn_det_pairs = sorted(tn_music_pair_hat.keys(), key=lambda k: (k[1], k[0]))

tn_music_pair_rx_idx_keep = np.asarray(tn_music_out["pair_rx_idx"], dtype=int)
tn_music_pair_t_keep = np.asarray(tn_music_out["pair_t_idx"], dtype=int)
tn_music_pair_bs_keep = np.asarray(tn_music_out["pair_bs_idx"], dtype=int)
tn_music_pair_sector_keep = np.asarray(tn_music_out["pair_sector_idx"], dtype=int)
tn_music_pair_phi_t_deg_keep = np.asarray(tn_music_out["pair_phi_hat_deg"], dtype=float)
tn_music_pair_theta_t_deg_keep = np.asarray(tn_music_out["pair_theta_hat_deg"], dtype=float)

tn_music_num_sources_record = np.asarray(tn_music_out["num_sources_record"], dtype=int)
tn_music_selected_metric_log = tn_music_out["selected_metric_log"]

if tn_music_num_sources_record.size > 0 and np.any(tn_music_num_sources_record >= 0):
    print("TN MUSIC estimated K (mean):", float(np.mean(tn_music_num_sources_record[tn_music_num_sources_record >= 0])))

for title, arr in [
    ("TN MUSIC hat mode selected", tn_music_out["selected_mode_record"]),
    ("TN MUSIC manifold selected", tn_music_out["selected_manifold_record"]),
    ("TN MUSIC flatten selected", tn_music_out["selected_flatten_record"]),
    ("TN MUSIC scan selected", tn_music_out["selected_scan_record"]),
]:
    if arr.size > 0:
        vals, cnts = np.unique(arr.astype(object), return_counts=True)
        msg = ", ".join([f"{v}:{int(c)}" for v, c in zip(vals, cnts)])
        print(f"{title}:", msg)
if np.asarray(tn_music_out["selected_phi_offset_record"]).size > 0:
    vals, cnts = np.unique(np.asarray(tn_music_out["selected_phi_offset_record"], dtype=float), return_counts=True)
    msg = ", ".join([f"{float(v):.1f}:{int(c)}" for v, c in zip(vals, cnts)])
    print("TN MUSIC phi-offset selected:", msg)

print("\nTN detected pairs (HAT):", len(tn_det_pairs))
theta_name = "elev" if str(theta_display_mode).lower() == "elevation" else "theta"
print(f"format: pair(rx,t,bs,sec,mode,mani,flat,scan,poff) | trueAOD_global(phi,{theta_name}) | hat_global(phi,{theta_name})")

for i, (rx, t) in enumerate(tn_det_pairs):
    rec = tn_music_pair_hat[(rx, t)]
    phi_hat = float(rec["phi_hat_deg"])
    theta_hat = float(rec["theta_hat_deg"])

    if str(theta_display_mode).lower() == "elevation":
        theta_hat_show = float(zenith_to_elevation_deg(np.array([theta_hat]))[0])
    else:
        theta_hat_show = float(theta_hat)

    phi_true, theta_true, _bs_t, _sec_t = tn_true_map[(rx, t)]
    if str(theta_display_mode).lower() == "elevation":
        theta_true_show = float(zenith_to_elevation_deg(np.array([theta_true]))[0])
    else:
        theta_true_show = float(theta_true)

    print(
        f"pair[{i:3d}] (rx={int(rx):3d}, t={int(t):2d}, bs={int(rec['bs'])}, sec={int(rec['sec'])}, mode={rec['hat_mode']}, mani={rec['manifold']}, flat={rec['flatten']}, scan={rec['scan']}, poff={float(rec['phi_offset_deg']):.1f}) | "
        f"trueAOD_global(phi={float(phi_true):7.2f}, {theta_name}={theta_true_show:7.2f}) | "
        f"hat_global(phi={phi_hat:7.2f}, {theta_name}={theta_hat_show:7.2f})"
    )

if len(tn_det_pairs) > 0:
    phi_true_match = np.array([tn_true_map[k][0] for k in tn_det_pairs], dtype=float)
    theta_true_match = np.array([tn_true_map[k][1] for k in tn_det_pairs], dtype=float)
    phi_hat_match = np.array([tn_music_pair_hat[k]["phi_hat_deg"] for k in tn_det_pairs], dtype=float)
    theta_hat_match = np.array([tn_music_pair_hat[k]["theta_hat_deg"] for k in tn_det_pairs], dtype=float)

    if str(theta_display_mode).lower() == "elevation":
        theta_true_metric = zenith_to_elevation_deg(theta_true_match)
        theta_hat_metric = zenith_to_elevation_deg(theta_hat_match)
        theta_metric_name = "elev"
    else:
        theta_true_metric = theta_true_match
        theta_hat_metric = theta_hat_match
        theta_metric_name = "theta"

    tn_metrics_aod = angle_error_metrics(
        phi_true_match,
        theta_true_metric,
        phi_hat_match,
        theta_hat_metric,
        reference_mode="aod",
    )

    print("\nTN Angle Error Summary on matched detected pairs:")
    print(
        "  ref=AOD(global) : "
        f"N={int(tn_metrics_aod['count'])}, "
        f"phi_MAE={tn_metrics_aod['phi_mae_deg']:.2f} deg, "
        f"{theta_metric_name}_MAE={tn_metrics_aod['theta_mae_deg']:.2f} deg"
    )


In [ ]:
# TN fixed-combo MUSIC sanity check (no auto-selection)
fixed_combo_label = "conj|xz:+1|F|phase_only|270.0"
fixed_combo_label = "conj|yz:+1|F|complex|0"

if "tn_true_map" not in globals() or "tn_pair_keys" not in globals():
    raise RuntimeError("Run the TN truth/pair preparation cell first.")

tn_fixed_combo_out = run_music_angle_pipeline(
    h_tn_all,
    tx_rows=int(tx_antenna_rows),
    tx_cols=int(tx_antenna_cols),
    nsect=int(globals().get("nsect_eff", getattr(SceneConfig, "nsect", nsect))),
    true_pair_map=tn_true_map,
    pair_keys=tn_pair_keys,
    detect_num_sources=globals().get("music_num_sources", None),
    detect_threshold=globals().get("music_threshold", 1.0),
    detect_user_powers=globals().get("music_user_powers", None),
    detect_noise_var=globals().get("music_noise_var", 0.0),
    detect_covariance_mode=globals().get("music_covariance_mode", "sample"),
    detect_num_snapshots=globals().get("music_num_snapshots", 100),
    detect_rng_seed=globals().get("music_rng_seed", None),
    detect_source_estimation=globals().get("music_source_estimation", "mdl"),
    detect_energy_ratio=globals().get("music_energy_ratio", 0.95),
    detect_reduce_rx_ant=globals().get("music_reduce_ntn_ant", "max"),
    hat_channel_mode="conj",
    ref_mode_for_auto=globals().get("music_ref_mode_for_auto", "aod"),
    auto_mode_min_pairs=globals().get("music_auto_mode_min_pairs", 1),
    use_sector_orientation=globals().get("music_use_sector_orientation", True),
    sector_pitch_rad=globals().get("music_sector_pitch_rad", -0.174533),
    sector_roll_rad=globals().get("music_sector_roll_rad", 0.0),
    rotation_order=globals().get("music_rotation_order", "zyx"),
    manifold_mode="yz:+1",
    manifold_auto_list=["yz:+1"],
    flatten_mode="F",
    flatten_mode_list=["F"],
    scan_mode="complex",
    scan_mode_list=["complex"],
    phi_offset_mode=0,
    phi_offset_list=[0],
    sector_forward_only=globals().get("music_sector_forward_only", True),
    sector_forward_cos_min=globals().get("music_sector_forward_cos_min", 0.0),
    phi_grid_deg=globals().get("music_phi_grid_deg", np.arange(0.0, 360.0, 2.0)),
    theta_grid_deg=globals().get("music_theta_grid_deg", np.arange(0.0, 181.0, 2.0)),
)

tn_fixed_pair_hat = tn_fixed_combo_out["pair_hat"]
tn_fixed_pairs = sorted(tn_fixed_pair_hat.keys(), key=lambda k: (k[1], k[0]))

print(f"TN fixed combo: {fixed_combo_label}")
print("Detected/kept TN links:", len(tn_fixed_pairs))

if len(tn_fixed_pairs) > 0:
    phi_true_match = np.array([tn_true_map[k][0] for k in tn_fixed_pairs], dtype=float)
    theta_true_match = np.array([tn_true_map[k][1] for k in tn_fixed_pairs], dtype=float)
    phi_hat_match = np.array([tn_fixed_pair_hat[k]["phi_hat_deg"] for k in tn_fixed_pairs], dtype=float)
    theta_hat_match = np.array([tn_fixed_pair_hat[k]["theta_hat_deg"] for k in tn_fixed_pairs], dtype=float)
    met = angle_error_metrics(
        phi_true_match,
        theta_true_match,
        phi_hat_match,
        theta_hat_match,
        reference_mode=globals().get("music_ref_mode_for_auto", "aod"),
    )
    theta_name = "elev" if str(theta_display_mode).lower() == "elevation" else "theta"
    print(
        f"Summary: total_N={int(met['count']):4d}, phi_MAE={met['phi_mae_deg']:7.2f} deg, "
        f"{theta_name}_MAE={met['theta_mae_deg']:7.2f} deg"
    )

    def _theta_disp(v):
        v = float(v)
        return 90.0 - v if str(theta_display_mode).lower() == "elevation" else v

    print("First 20 TN links (fixed combo):")
    for i, (rx, t) in enumerate(tn_fixed_pairs[:20]):
        rec = tn_fixed_pair_hat[(rx, t)]
        phi_true, theta_true, bs_true, sec_true = tn_true_map[(rx, t)]
        print(
            f"pair[{i:2d}] rx={int(rx):3d}, t={int(t):2d}, (bs={int(bs_true)}, sec={int(sec_true)}) | "
            f"phi_hat={float(rec['phi_hat_deg']):7.2f}, phi_true={float(phi_true):7.2f}, "
            f"theta_hat={_theta_disp(rec['theta_hat_deg']):7.2f}, theta_true={_theta_disp(theta_true):7.2f}"
        )
else:
    print("No TN links available for fixed-combo evaluation.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# background
colors = ['lightgray', 'brown']
cmap_bg = ListedColormap(colors)

extent = SceneConfig.extent
point_type = SceneConfig.point_type

tn_bs_index = SceneConfig.tn_bs_index

plt.figure()
plt.imshow(point_type, cmap=cmap_bg, interpolation='nearest', extent=extent)


# BS colors: high-contrast palette
cmap_bs = plt.get_cmap("tab10")
n_bs = tx_pos.shape[0]
bs_colors = [cmap_bs(i % 10) for i in range(n_bs)]


# BS triangles (bigger and more visible)
bs_marker = (3, 0, -30)
for i in range(n_bs):
    plt.scatter(
        tx_pos[i, 0], tx_pos[i, 1],
        c=[bs_colors[i]],
        marker=bs_marker,
        s=180,           # bigger
        linewidths=1.5,  # thicker edge
        # edgecolors='k',  # black edge for contrast
        label=f"BS {i}"
    )


# TN points (same as compute_positions: green x)
plt.scatter(tn_pos[:, 0], tn_pos[:, 1], color='green', marker='x', s=10, label='TN')

# NTN points (same as compute_positions: blue x)
plt.scatter(left_ntn_pos[:, 0], left_ntn_pos[:, 1], color='blue', marker='x', s=5, label='NTN')

# Outer circles on TN, color matched to its BS
for i in range(tn_pos.shape[0]):
    bs_id = int(tn_bs_index[i])
    plt.scatter(
        tn_pos[i, 0], tn_pos[i, 1],
        edgecolors=bs_colors[bs_id], facecolors='none',
        marker='o', s=100, linewidths=1.5
    )

# plt.title('Spatial Distribution of TN/NTN and BSs')
plt.title('Spatial Distribution of TN/NTN UEs and BSs')

plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.02), fontsize=10, frameon=True)

import os
# save_path = "/home/sj4025/my_project/ntn_int_nulling/map.png"
# os.makedirs(os.path.dirname(save_path), exist_ok=True)
# plt.savefig(save_path, dpi=300, format='png')
plt.show()


Different Lambda_

In [ ]:
# Create a new figure
plt.figure(figsize=(8, 6))

# ------- Plot raw INR (no nulling) -------
inr_array = np.array(inr_list)
inr_array = inr_array[np.isfinite(inr_array)]
inr_sorted = np.sort(inr_array)
if len(inr_sorted) > 0:
    cdf_inr = np.arange(1, len(inr_sorted) + 1) / len(inr_sorted)
    plt.plot(inr_sorted, cdf_inr, label=f"Raw INR (n={len(inr_sorted)})", linewidth=2, color='black')

# ------- Plot INR Nulling for each lambda -------
for lambda_ in lambda_ranges:
    raw_inr_null_list = inr_nulling_dict[lambda_]
    inr_null_array = np.array(raw_inr_null_list)
    inr_null_array = inr_null_array[np.isfinite(inr_null_array)]
    inr_null_sorted = np.sort(inr_null_array)

    if len(inr_null_sorted) > 0:
        cdf_inr_null = np.arange(1, len(inr_null_sorted) + 1) / len(inr_null_sorted)
        plt.plot(inr_null_sorted, cdf_inr_null,
                #  label=f"Nulling INR (λ={lambda_:.0e}, n={len(inr_null_sorted)})",
                label=f"Nulling INR (λ={lambda_:.0e})",
                 linestyle="--", linewidth=2)

# ------- Make the plot pretty -------
plt.xlabel("INR (dB)")
plt.ylabel("CDF")
plt.title("CDF of Raw INR vs. Perfect Nulling INR for Different λ Values")
plt.grid(True)
plt.legend()
plt.ylim([0.01, 1])
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))
# ------- Plot raw SNR (no nulling) -------
snr_array = np.array(snr_list)
snr_array = snr_array[np.isfinite(snr_array)]
snr_sorted = np.sort(snr_array)

if len(snr_sorted) > 0:
    cdf_snr = np.arange(1, len(snr_sorted) + 1) / len(snr_sorted)
    plt.plot(snr_sorted, cdf_snr, label=f"Raw SNR (n={len(snr_sorted)})", linewidth=2, color='black')

# ------- Plot Nulling SNR for each lambda -------
for lambda_ in lambda_ranges:
    # Ideal interference known
    snr_null = np.array(snr_nulling_dict[lambda_])
    snr_null = snr_null[np.isfinite(snr_null)]
    snr_null_sorted = np.sort(snr_null)

    if len(snr_null_sorted) > 0:
        cdf_snr_null = np.arange(1, len(snr_null_sorted) + 1) / len(snr_null_sorted)
        plt.plot(snr_null_sorted, cdf_snr_null,
                 label=f"Perfect Nulling (λ={lambda_:.0e})", linestyle='--', linewidth=2)


# ------- Make the plot nice -------
plt.xlabel("SNR (dB)")
plt.ylabel("CDF")
plt.title("CDF of Raw vs. Perfect Nulling SNR under Different λ Values")
plt.grid(True)
plt.legend()
plt.ylim([0, 1])
plt.tight_layout()
plt.show()


Different Satellite Angle

In [ ]:
# CODEx_FIXED: radiomap render function (no npz required)
import mitsuba as mi
from sionna.rt import RadioMapSolver, Camera, Receiver
import numpy as np
import matplotlib.pyplot as plt

try:
    from sionna.rt import transform_mesh
except Exception:
    transform_mesh = None


def _get_terrain_mesh(scene, terrain_name="Terrain", z_offset=0.1):
    if hasattr(scene, "objects") and terrain_name in scene.objects:
        terrain = scene.objects[terrain_name].clone(as_mesh=True)
    else:
        terrain = None
        if hasattr(scene, "objects"):
            for name in scene.objects.keys():
                if "terrain" in name.lower() or "plane" in name.lower():
                    terrain = scene.objects[name].clone(as_mesh=True)
                    break
        if terrain is None:
            return None
    if transform_mesh is not None:
        transform_mesh(terrain, translation=[0, 0, z_offset])
    else:
        raise RuntimeError("transform_mesh not available in your sionna version")
    return terrain


def _build_precoding_from_record(scene, nsect, lambda_):
    if lambda_ not in v_null_record or len(v_null_record[lambda_]) == 0:
        raise ValueError(f"No v_null_record for lambda={lambda_}")

    t_to_v = {}
    for t, v in zip(v_null_t_index[lambda_], v_null_record[lambda_]):
        t = int(t)
        if t not in t_to_v:
            t_to_v[t] = np.asarray(v).reshape(-1).astype(np.complex64)

    tx_names = list(scene.transmitters)
    if len(tx_names) == 0:
        raise ValueError("scene.transmitters is empty. Run compute_paths first.")

    num_tx_ant = next(iter(t_to_v.values())).shape[0]
    precoding = np.zeros((len(tx_names), num_tx_ant), dtype=np.complex64)

    for i, name in enumerate(tx_names):
        if not name.startswith("tx-"):
            continue
        parts = name.split("-")
        if len(parts) != 3:
            continue
        try:
            bs_idx = int(parts[1])
            sec_idx = int(parts[2])
        except ValueError:
            continue
        t = bs_idx * nsect + sec_idx
        if t in t_to_v:
            precoding[i, :] = t_to_v[t]

    return precoding


def _cast_solver_seed(seed):
    if seed is None:
        return None
    for type_name in ("UInt", "UInt32", "UInt64"):
        t = getattr(mi, type_name, None)
        if t is not None:
            try:
                return t(int(seed))
            except Exception:
                pass
    try:
        import drjit as dr
        return dr.uint32(int(seed))
    except Exception:
        return int(seed)




def _patch_sampler_seed_compat(solver):
    """Patch sampler.seed to support both (seed) and (seed, wavefront_size)."""
    sampler = getattr(solver, "_sampler", None)
    if sampler is None:
        return
    cls = sampler.__class__
    if getattr(cls, "_codex_seed_patched", False):
        return

    orig_seed = getattr(cls, "seed", None)
    if orig_seed is None:
        return

    def seed_compat(self, seed, wavefront_size=4294967295):
        try:
            return orig_seed(self, seed, int(wavefront_size))
        except TypeError:
            return orig_seed(self, seed)

    try:
        cls.seed = seed_compat
        cls._codex_seed_patched = True
    except Exception:
        pass


def render_nulling_radiomap_all(scene, nsect, lambda_,
                                center=None, size=None, cell_size=(10, 10),
                                max_depth=3, samples_per_tx=2**18,
                                metric="inr_ntn",
                                show_scene=False,
                                show_2d=False,
                                use_terrain=True,
                                terrain_name="Terrain",
                                terrain_z_offset=1.5,
                                show_tn=False,
                                show_left_ntn=False,
                                show_legend=False,
                                legend_loc="upper right",
                                tn_color=(0.0, 0.35, 1.0),
                                left_ntn_color=(0.0, 1.0, 1.0),
                                tn_display_radius=120,
                                left_ntn_display_radius=90,
                                rm_vmin=None, rm_vmax=None,
                                camera_pos=[0, 0, 9000],
                                camera_look_at=[0, 0, 0],
                                render_resolution=(5000, 5000),
                                render_fov=95,
                                render_num_samples=16,
                                render_retry_on_oom=True,
                                seed=1,
                                precoding_matrix=None):

    for rx_name in list(scene.receivers):
        scene.remove(rx_name)

    tx_names = list(scene.transmitters)
    if len(tx_names) == 0:
        raise ValueError("scene.transmitters is empty. Run compute_paths first.")

    if precoding_matrix is not None:
        precoding = np.asarray(precoding_matrix, dtype=np.complex64)
        if precoding.ndim != 2:
            raise ValueError(f"precoding_matrix must be 2-D, got shape {precoding.shape}")
        if precoding.shape[0] != len(tx_names):
            raise ValueError(f"precoding_matrix row mismatch: got {precoding.shape[0]}, expected {len(tx_names)}")
    else:
        precoding = _build_precoding_from_record(scene, nsect, lambda_)

    nz = np.linalg.norm(precoding, axis=1) > 0
    print(f"TX total: {len(tx_names)}, with precoding: {np.count_nonzero(nz)}")
    if np.count_nonzero(nz) == 0:
        raise ValueError("All precoding vectors are zero.")

    precoding_vec = (
        mi.TensorXf(np.ascontiguousarray(precoding.real.astype(np.float32))),
        mi.TensorXf(np.ascontiguousarray(precoding.imag.astype(np.float32))),
    )

    rm_solver = RadioMapSolver()
    # Mitsuba sampler requires wavefront_size <= 2^32-1.
    num_tx = len(tx_names)
    max_wavefront = (2**32) - 1
    max_samples_per_tx = max(1, max_wavefront // max(1, num_tx))
    eff_samples_per_tx = int(min(int(samples_per_tx), int(max_samples_per_tx)))
    if eff_samples_per_tx != int(samples_per_tx):
        print(f"samples_per_tx clipped from {int(samples_per_tx)} to {eff_samples_per_tx} for {num_tx} TX")

    _patch_sampler_seed_compat(rm_solver)
    seed_cast = _cast_solver_seed(seed)

    measurement_surface = None
    if use_terrain:
        measurement_surface = _get_terrain_mesh(scene, terrain_name=terrain_name, z_offset=terrain_z_offset)

    if measurement_surface is not None:
        rm = rm_solver(scene,
                       measurement_surface=measurement_surface,
                       max_depth=max_depth,
                       samples_per_tx=eff_samples_per_tx,
                       precoding_vec=precoding_vec,
                       cell_size=cell_size,
                       seed=seed_cast)
    else:
        if center is not None:
            center = [float(center[0]), float(center[1]), float(center[2])]
        if size is not None:
            size = [float(size[0]), float(size[1])]

        rm = rm_solver(scene,
                       max_depth=max_depth,
                       samples_per_tx=eff_samples_per_tx,
                       precoding_vec=precoding_vec,
                       cell_size=cell_size,
                       center=center,
                       size=size,
                       orientation=[0, 0, 0],
                       seed=seed_cast)

    if show_scene:
        temp_names = []
        if show_tn and 'tn_pos' in globals():
            for i, pos in enumerate(tn_pos):
                name = f"tmp-tn-{i}"
                scene.add(Receiver(name=name, position=pos, color=tuple(tn_color), display_radius=tn_display_radius))
                temp_names.append(name)
        if show_left_ntn and 'left_ntn_pos' in globals() and left_ntn_pos is not None:
            for i, pos in enumerate(left_ntn_pos):
                name = f"tmp-leftntn-{i}"
                scene.add(Receiver(name=name, position=pos, color=tuple(left_ntn_color), display_radius=left_ntn_display_radius))
                temp_names.append(name)

        cam_dist = float(np.linalg.norm(np.asarray(camera_pos, dtype=float) - np.asarray(camera_look_at, dtype=float)))
        if cam_dist > 9800:
            print(f"warning: camera distance {cam_dist:.1f}m is close to/exceeds Sionna far_clip=10000; image may be blank")
        cam = Camera(position=list(camera_pos), look_at=list(camera_look_at))
        render_kwargs = dict(camera=cam,
                             radio_map=rm,
                             resolution=tuple(render_resolution),
                             fov=float(render_fov),
                             rm_show_color_bar=True,
                             rm_vmin=rm_vmin,
                             rm_vmax=rm_vmax,
                             rm_metric=metric,
                             num_samples=int(render_num_samples))
        try:
            scene.render(**render_kwargs)
        except RuntimeError as e:
            msg = str(e).lower()
            if (not render_retry_on_oom) or ("out of memory" not in msg):
                raise
            w, h = render_kwargs["resolution"]
            ns = int(render_kwargs["num_samples"])
            w2, h2 = max(1024, w//2), max(1024, h//2)
            ns2 = max(4, ns//2)
            print(f"render OOM: retry with resolution={(w2, h2)}, num_samples={ns2}")
            render_kwargs["resolution"] = (w2, h2)
            render_kwargs["num_samples"] = ns2
            scene.render(**render_kwargs)

        if show_legend:
            from matplotlib.lines import Line2D

            handles = [
                Line2D([0], [0], marker='o', linestyle='None', color='red',
                       markerfacecolor='red', markeredgecolor='red', markersize=8, label='BS')
            ]
            if show_tn:
                handles.append(
                    Line2D([0], [0], marker='o', linestyle='None', color=tuple(tn_color),
                           markerfacecolor=tuple(tn_color), markeredgecolor=tuple(tn_color),
                           markersize=8, label='TN')
                )
            if show_left_ntn:
                handles.append(
                    Line2D([0], [0], marker='o', linestyle='None', color=tuple(left_ntn_color),
                           markerfacecolor=tuple(left_ntn_color), markeredgecolor=tuple(left_ntn_color),
                           markersize=8, label='NTN')
                )

            try:
                fig = plt.gcf()
                ax = fig.axes[0] if len(fig.axes) > 0 else plt.gca()
                if legend_loc == 'upper left':
                    ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(0.02, 0.98),
                              borderaxespad=0.0, frameon=True)
                else:
                    ax.legend(handles=handles, loc=legend_loc, frameon=True)
                plt.show()
            except Exception:
                fig_leg, ax_leg = plt.subplots(figsize=(4.8, 0.8))
                ax_leg.axis('off')
                ax_leg.legend(handles=handles, loc='center', ncol=len(handles), frameon=True)
                plt.show()

        for name in temp_names:
            if name in scene.receivers:
                scene.remove(name)
                
    return rm






In [ ]:
# # CODEx_FIXED: direct render from in-memory v_null_record (no npz)
# import inspect

# src_check = inspect.getsource(render_nulling_radiomap_all)
# print('render has seed compat patch:', '_patch_sampler_seed_compat' in src_check)
# seed_obj = _cast_solver_seed(1)
# print('seed cast type:', type(seed_obj))

# render_nulling_radiomap_all(
#     scene, nsect, lambda_=1e5,
#     metric='inr_ntn',
#     show_scene=True,
#     use_terrain=False,
#     center=[0, 0, 100],
#     size=[10000, 10000],
#     cell_size=(10, 10),
#     max_depth=3,
#     samples_per_tx=10**8,
#     rm_vmin=-20, rm_vmax=45,
#     show_tn=True,
#     show_left_ntn=True,
#     seed=1
# )




In [ ]:
rm = render_nulling_radiomap_all(
    scene, nsect, lambda_=1e5,
    metric='inr_ntn',
    show_scene=True,
    show_legend=True,
    legend_loc='upper left',
    use_terrain=True,
    terrain_name="terrain",
    terrain_z_offset=2.5,
    cell_size=(5, 5),
    max_depth=3,
    samples_per_tx=10*10**7,
    show_tn=True,
    show_left_ntn=True,
    tn_display_radius=140,
    left_ntn_display_radius=110,
    camera_pos=[0, 0, 5000],
    camera_look_at=[0, 0, 0],
    render_resolution=(9000, 9000),
    render_fov=90,
    rm_vmin=-25,
    rm_vmax=45,
    seed=1
)


In [ ]:
scene.preview(
    paths= SceneConfig.paths_ntn,
    radio_map=rm,
    rm_metric="inr_ntn",   # 或 "rss" / "sinr"
    rm_vmin=-25,
    rm_vmax=45
    # rm_db_scale=True
)